In [ ]:
import dijido
dijido.login()

In [ ]:
# Get all activities that are part of a specific goal name. Get

In [ ]:
GOAL_NAME = "Check miscellaneous contact streams occasionally"

In [ ]:
goals = dijido.get_goals_by_name(GOAL_NAME)

In [ ]:
if len(goals) != 1:
    raise ValueError(f"Expecting 1 goal, got {len(goals)}")

In [ ]:
activities = dijido.get_activities_for_goal(goals[0]["_id"])

In [ ]:
import copy

activities_raw = copy.deepcopy(activities)

In [ ]:
df_activities.sort_values(by="duration", ascending=False)

In [ ]:
import pandas as pd
from datetime import datetime, timedelta

# Convert start_time and end_time to datetime objects and calculate duration in minutes
activities = copy.deepcopy(activities_raw)

for activity in activities:
    activity['start_time'] = datetime.fromisoformat(activity['start_time'][:-1])
    activity['end_time'] = datetime.fromisoformat(activity['end_time'][:-1])
    activity['duration'] = (activity['end_time'] - activity['start_time']).total_seconds() / 60

# Create a dataframe from the activities list
df_activities = pd.DataFrame(activities)

# Create a date range for the 2-week periods
start_date = df_activities['start_time'].min().date()
end_date = df_activities['end_time'].max().date()
date_range = pd.date_range(start=start_date, end=end_date, freq='14D')

# Initialize a list to store the results
results = []
cumsum_results = []

cumsum = 0

# Calculate the 14-day moving average for each period
for start in date_range:
    end = start + timedelta(days=14)
    mask = (df_activities['start_time'].dt.date < end) & (df_activities['end_time'].dt.date >= start)
    period_activities = df_activities[mask]
    
    total_minutes = 0
    for _, row in period_activities.iterrows():
        activity_start = max(row['start_time'], start)
        activity_end = min(row['end_time'], end)
        duration = (activity_end - activity_start).total_seconds() / 60
        total_minutes += duration
        cumsum += duration
        cumsum_results.append({'day': activity_end, 'cumsum': cumsum})
    
    avg_minutes_per_day = total_minutes / 14
    results.append({'start_day': start, '14_day_avg_minutes_per_day': avg_minutes_per_day})

# Create the final dataframe
df_results = pd.DataFrame(results)
print(df_results)

In [ ]:
# Display any activities with negative duration
mask = df_activities['duration'] < 0

if mask.any():
    print("\nActivities with negative duration:")
    print(df_activities[mask])

In [ ]:
import plotly.express as px

# Create a line chart using Plotly
fig = px.line(df_results, x='start_day', y='14_day_avg_minutes_per_day', title='14-Day Average Minutes Per Day')

# Show the plot
fig.show()

In [ ]:
import plotly.express as px

df_results = pd.DataFrame(cumsum_results)
print(df_results)

# Create a line chart using Plotly
fig = px.line(df_results, x='day', y='cumsum', title='14-Day Average Minutes Per Day')

# Show the plot
fig.show()

In [ ]:
import ruptures as rpt

signal = df_results[df_results["day"] > datetime(2024, 1, 23)]["cumsum"].values

algo = rpt.Pelt(model="rbf").fit(signal)
result = algo.predict(pen=10)

# display
rpt.display(signal, result)
plt.show()